# EECE 5554 Lab 2: RTK GNSS

## Learning objectives

Lab 2 builds directly on Lab 1. You will feed the same kind of receiver a stream of
corrections from a reference station 13 km away, collect matched data with and without
those corrections, and quantify how much the fix tightens.

By the end of this lab a well-prepared student will be able to:
<ul>
    <li> Explain what a differential correction contains and which error sources it removes </li>
    <li> Distinguish an RTK float solution from an RTK fixed solution, and say what is being resolved </li>
    <li> Configure a Septentrio mosaic-X5 as an NTRIP client and verify that corrections are being used </li>
    <li> Extend a custom ROS 2 message and driver to carry solution quality alongside position </li>
    <li> Quantify precision with centroid deviation, standard deviation, and 2DRMS </li>
    <li> Explain why this lab measures precision and not accuracy, and what would be required to measure accuracy </li>
</ul>

## Prerequisites

| Item | Explain |
|---|---|
| Lab 1 | Complete and working. You reuse the parser, the UTM conversion, and most of the driver |
| PA0, PA1 | Ubuntu 24.04 and ROS 2 Jazzy, as before |
| Hardware | SparkFun RTK mosaic-X5 kit, issued at the classroom test and returned after the field session |

## Software to install

```bash
pip3 install --user numpy matplotlib
```

You should already have `utm` from Lab 1. Nothing else is needed. In particular you do
**not** need RTKLIB, `str2str`, or any NTRIP software on your laptop, because the
receiver fetches its own corrections.

Make sure the course repository is current:

```bash
cd ~/5554-F26 && git pull
```

## Individual and team work

**Individual:** write your own driver, run your own analysis, write your own answers.
**Shared:** one receiver per team in the classroom and in the field. Sharing the raw
rosbags with your teammates is expected. Everything downstream of the bags is yours.

## Collaboration statement

You should complete this assignment without collaboration with other students in the
course or elsewhere, beyond the shared hardware and shared raw data noted above. Please
ask for help from the TAs or course instructors if you are getting stuck.

## Background: the five minutes you actually need

Lecture 05 covers the theory. This is enough to run the lab.

**Why corrections work.** Most of the error in a standalone fix comes from satellite
clock and orbit error and from the atmosphere, not from your receiver. A reference
station a few kilometres away sees almost the same error at the same instant, broadcasts
what it sees, and your receiver subtracts it.

**What does not get removed.** Multipath and receiver noise are local to your antenna.
This is why the lab has an occluded run.

**Float and fixed.** RTK uses the carrier phase, which is precise but ambiguous by a
whole number of 19 cm cycles. Working those integers out is ambiguity resolution.

| State | Meaning | Typical horizontal precision |
|---|---|---|
| Standalone | No corrections applied | 1 to 3 m |
| RTK float | Corrections in use, integers unresolved | 0.2 to 1 m |
| RTK fixed | Integers resolved | 0.01 to 0.03 m |

The gap between float and fixed is much larger than the gap between standalone and
float. That is the main thing to notice in your data.

**How you read the state.** Field 6 of every GGA sentence, the quality indicator.

| Value | Meaning |
|---|---|
| 0 | No solution. Drop the epoch |
| 1 | Standalone |
| 2 | Differential |
| 4 | **RTK fixed** |
| 5 | **RTK float** |

**Where our corrections come from.** Two volunteer reference stations published on
RTK2go, a free community NTRIP caster. Your receiver connects over your team's phone
hotspot and pulls corrections directly, with nothing in between.

| | Primary | Backup |
|---|---|---|
| Caster | `rtk2go.com` port `2101` | same |
| Mountpoint | `BI_WARD_SANGALA` | `LFPWD_HV` |
| Location | Bainbridge Island | Lake Forest Park |
| Distance from Lake Union Park | about 13 km | about 16.5 km |

These are private receivers that volunteers have chosen to share. They can go offline
without notice and nobody owes you an explanation, which is why you configure both.

## Step 1: Create the two packages

Same structure as Lab 1, in a new workspace. A ROS 2 message still cannot be generated
from an `ament_python` package, so there are still two packages.

```bash
cd ~/EECE5554                       # your private repo from PA0
mkdir -p Lab2/ros2_ws/src
cd Lab2/ros2_ws/src

ros2 pkg create --build-type ament_cmake gps_msgs
ros2 pkg create --build-type ament_python --dependencies rclpy std_msgs gps_msgs gps_driver
```

Target layout:

```
Lab2/ros2_ws/src/
├── gps_msgs/
│   ├── CMakeLists.txt
│   ├── package.xml
│   └── msg/
│       └── Customrtk.msg
└── gps_driver/
    ├── gps_driver/
    │   ├── __init__.py
    │   └── rtk_driver.py
    ├── launch/
    │   └── rtk_driver.launch.py
    ├── package.xml
    ├── setup.cfg
    └── setup.py
```

**Names matter.** The self-check and the grader look for exactly `gps_msgs`,
`gps_driver`, `Customrtk.msg`, and `rtk_driver.launch.py`. Capitalization counts.

The package names are the same as Lab 1 on purpose. Copy your Lab 1 `CMakeLists.txt`,
`package.xml`, `setup.py`, and `setup.cfg` across and change only what this lab needs.

## Step 2: Define the custom message

Create `gps_msgs/msg/Customrtk.msg`. It is your Lab 1 `Customgps.msg` with **three**
fields added at the end. Note that `hdop` was already there, so do not add it twice.

```
std_msgs/Header header
float64 latitude
float64 longitude
float64 altitude
float64 utm_easting
float64 utm_northing
uint8 zone
string letter
float64 hdop
string gpgga_read
uint8 fix_quality
uint8 num_satellites
float32 correction_age
```

| New field | Source | Notes |
|---|---|---|
| `fix_quality` | GGA field 6 | The quality indicator. This is the whole point of the lab |
| `num_satellites` | GGA field 7 | Satellites used in the solution |
| `correction_age` | GGA field 14 | Seconds since the last correction was applied |

Update `rosidl_generate_interfaces` in `CMakeLists.txt` to list `msg/Customrtk.msg`
instead of `msg/Customgps.msg`. Everything else in that file stays the same.

Build and confirm:

```bash
cd ~/EECE5554/Lab2/ros2_ws
colcon build --symlink-install
source install/setup.bash
ros2 interface show gps_msgs/msg/Customrtk
```

If that command does not print your definition, nothing downstream will work. Fix it now.

## Step 3: Write the driver

Copy `standalone_driver.py` from Lab 1 to
`gps_driver/gps_driver/rtk_driver.py` and modify it. Four changes:

1. **Publish `Customrtk`** instead of `Customgps`, still on the topic `/gps`
2. **Accept `$GNGGA` as well as `$GPGGA`.** The mosaic-X5 tracks GPS, GLONASS, Galileo
   and BeiDou together, so it emits `$GNGGA`. A driver that only matches `$GPGGA` will
   sit silent and look broken
3. **Parse the three new fields** out of the same sentence you already split
4. **Take the port as a parameter** as before, at **115200 baud**, not the 4800 you
   used for the Lab 1 puck

Everything else carries over unchanged: hemisphere handling, the UTM conversion,
`header.frame_id` as `GPS1_Frame`, UTC from the sentence rather than the system clock,
and `gpgga_read` holding the raw sentence.

### The empty field trap

```
$GNGGA,181638.00,4737.2891,N,12220.2355,W,1,08,1.42,41.2,M,-17.5,M,,0000*7A
                                                                   ^^ empty
```

With corrections off there is no correction age to report, so that field is **empty,
not zero**. `float('')` raises `ValueError`, your node dies partway through Run A, and
you lose the run. Satellite count and HDOP can also arrive empty when there is no fix.

```python
def to_float(field, default=0.0):
    # Parse an NMEA field that may be empty.
    field = field.strip()
    return float(field) if field else default
```

Use a helper like this for every numeric field, and wrap the parse so a single bad
sentence cannot kill the node:

```python
try:
    msg = self.parse_gga(line)
except (ValueError, IndexError) as e:
    self.get_logger().warn(f"skipped a bad sentence: {e}")
    return
```

Your Lab 1 checksum validation still applies and is still worth keeping.

## Step 4: Write the launch file

Create `gps_driver/launch/rtk_driver.launch.py`. Same structure as Lab 1, same `port`
argument:

```bash
ros2 launch gps_driver rtk_driver.launch.py port:=/dev/ttyACM0
```

Remember the `data_files` entry in `setup.py` that installs the `launch` directory, or
`ros2 launch` will not find the file.

## Step 5: Test against the emulator

Develop the whole driver before you touch the receiver, exactly as in Lab 1.

```bash
cd ~/5554-F26/sensor_emulator
python3 serial_emulator.py -f data/rtk_seattle.txt
```

It prints a port name such as `/dev/pts/3`. In another terminal:

```bash
cd ~/EECE5554/Lab2/ros2_ws
source install/setup.bash
ros2 launch gps_driver rtk_driver.launch.py port:=/dev/pts/3
```

In a third:

```bash
ros2 topic echo /gps --field fix_quality
```

`rtk_seattle.txt` is 300 sentences over five minutes, and the solution state walks
through all three values at fixed times:

| Epochs | `fix_quality` | State | Correction age field |
|---|---|---|---|
| 0 to 59 | `1` | Standalone | **empty** |
| 60 to 89 | `5` | RTK float | 0.8 to 2.4 s |
| 90 to 299 | `4` | RTK fixed | 0.6 to 2.2 s |

So time to first float is 60 s and time to first fix is 90 s. Those are known answers
you can check your own timing calculation against.

There is a second vector, `data/rtk_occluded.txt`, where the state flaps instead of
settling and eight epochs carry quality `0` with empty latitude, longitude, satellite
count and HDOP. It exists so that a driver which survives the first vector still has
something to fail on. Your node must not crash on it.

### Run the self-check

```bash
cd ~/5554-F26/sensor_emulator
./verify_lab2.sh <your-northeastern-username>
```

It checks your package layout, builds the workspace, confirms all thirteen fields are
present in `Customrtk.msg`, runs your driver against **both** vectors, and compares
every published field against values recomputed from the sentences. Messages are
matched to sentences by `gpgga_read`, so a driver that drops epochs is reported rather
than silently passing.

It takes about three minutes and writes `<username>_lab2.txt` into your `Lab2` folder.
Push that file whether it passes or not.

**Checkpoint.** All four must hold before the classroom test:

<ul>
    <li> Messages appear on <code>/gps</code> </li>
    <li> Latitude and longitude are sensible, UTM easting and northing are populated </li>
    <li> <code>fix_quality</code> <b>changes value</b> as the vector replays </li>
    <li> The node survives a sentence with empty fields </li>
</ul>

Bring a laptop that passes all four to the classroom test. We cannot debug fifteen
parsers and configure fifteen receivers in the same session.

## Step 6: Configure the receiver in the classroom

We do this together in class. Full step-by-step instructions are in
**`EECE5554_Lab2_clinic_handout.pdf`**. Print it and bring it. The short version, so you
know what you are walking into:

The mosaic-X5 runs its own NTRIP client, so the correction stream never touches your
laptop. The receiver has two independent connections and they do different jobs.

| Connection | Carries | Address |
|---|---|---|
| Receiver to **your team's phone hotspot**, over wifi via its built-in ESP32 | RTCM corrections from RTK2go | given by your hotspot |
| Receiver to your laptop, over USB-C | Web interface, and NMEA out to your driver | `192.168.3.1` |

Because the receiver fetches its own corrections over your own hotspot, **your setup is
completely self-contained.** It works in the classroom, at Lake Union Park, and anywhere
else you have phone signal. Nothing depends on the instructor being present.

Eight steps, which we will work through together:

1. Antenna on, power up over USB-C, wait 60 seconds
2. Browse to `http://192.168.3.1`
3. Turn on your phone's personal hotspot and join the receiver to it, following the
   steps on the Lecture 05 slides
4. Corrections tab, New NTRIP client:

   | Field | Value |
   |---|---|
   | Caster | `rtk2go.com` |
   | Port | `2101` |
   | Mount point | pick `BI_WARD_SANGALA` from the dropdown |
   | User name | **your own email address** |
   | Password | `none` |
   | Mode | On |

5. Add a second client the same way, picking `LFPWD_HV`
6. Enable GGA out at 1 Hz on `[[ MOSAIC_USB_PORT ]]` at 115200 baud, and turn on SD
   card logging
7. **Save the configuration**, then power cycle and confirm it survived
8. Verify: two green lines in the Corrections tab, and `fix_quality` moving from 1 to 5
   on your `/gps` topic

Enter the caster and port **first**. Once you do, the receiver fetches the caster's list
of streams and **Mount point** becomes a dropdown. You are picking from a menu rather
than typing a case-sensitive name, which is where most first attempts fail.

RTK2go requires a valid email address in the user name field. It is how they contact you
if your client is misbehaving. Use your real Northeastern address, not a made-up one.

> **The rule that matters.** RTK2go bans by IP address when a client reconnects
> abusively or when someone points a web browser at the caster. Browsing to your
> receiver at `192.168.3.1` is fine and you will do it constantly. Browsing to
> `rtk2go.com` is not, and it will lock **your** hotspot out for about three hours,
> usually on the afternoon you were planning to collect data. If a connection fails,
> fix the settings rather than hammering retry.

Indoors you will probably reach float and not fixed. **That is a pass.** You are
verifying that the chain works, not that the position is good. Both of those come
outdoors, so do not start changing settings because you are not seeing a 4.

## Step 7: Collect five data sets

Two outings. The first is with the class, the second is on your own schedule.

| | When | What | Where |
|---|---|---|---|
| **Outing 1** | Lake Union Park, **Monday 28 September or Thursday 1 October**, depending on equipment setup and weather. Confirmed in class and on Canvas | `open_standalone`, `open_rtk`, `walking_rtk` | We meet in the **classroom** and walk over together. Do not go straight to the park |
| **Outing 2** | Any time that week, before the deadline | `occluded_standalone`, `occluded_rtk` | Anywhere with a building face and a markable spot. Campus works |

Outing 2 is mechanically identical to what you will already have done once, so you do
not need an instructor present. Your receiver uses your own hotspot, so nothing
constrains when you go.

### The five runs

| Bag name | Site | Corrections | Duration | Outing |
|---|---|---|---|---|
| `open_standalone` | Open sky | Mode **off** | 10 min | 1 |
| `open_rtk` | Open sky | Mode **on** | 10 min | 1 |
| `walking_rtk` | Straight path, 200 to 300 m | Mode **on** | one pass | 1 |
| `occluded_standalone` | Near a building face | Mode **off** | 5 min | 2 |
| `occluded_rtk` | Near a building face | Mode **on** | 5 min | 2 |

```bash
cd ~/EECE5554/Lab2/data
ros2 bag record /gps -o open_standalone
```

### The sites

**Open sky (outing 1).** The lawn and pier on the north side of Lake Union Park, facing
the water. Water to the north means nothing reflects signals back from that direction,
and the horizon is clear in most directions. About as good as a city site gets.

**Occluded (outing 2).** Stand within **2 m** of a building face, with roughly half the
sky blocked, so the facade reflects signals straight back at your antenna. Glass and
flat concrete are worse than brick, which is what you want. Record where you stood and
photograph it.

### How you toggle corrections

Web interface, Corrections tab, edit icon on each NTRIP client, set **Mode** to off or
on. Both clients, every time. Nothing physical changes, no cables are unplugged, and
the antenna does not move.

### The one rule

**The antenna does not move between `open_standalone` and `open_rtk`, or between
`occluded_standalone` and `occluded_rtk`.** Mark the spot with tape. Toggling Mode is
the only difference between the two runs in a pair. If the tripod gets bumped, say so in
your field log and redo the pair.

### Procedure, both outings

1. Set up the tripod. Mark the spot. Note the antenna height
2. Photograph the site from the antenna position facing north, east, south and west.
   Put these in `Lab2/photos/`. You need them for a question later
3. Both NTRIP clients Mode **off**, confirm `fix_quality` reads 1, record the standalone
   run for the stated duration
4. Without touching the tripod, set both clients Mode **on**. **Write down** the wall
   clock time, the time `fix_quality` first reads 5, and the time it first reads 4.
   Once it is stable, record the RTK run
5. On outing 1 only: walk a straight 200 to 300 m path with corrections on, recording
   `walking_rtk`. Note your start and end points. Walk at a steady pace and do not stop

**A run that never reaches fixed is a result, not a failure.** If `occluded_rtk` sits at
float for the whole five minutes, record it and move on. Do not extend the run hoping
for a fix. That behaviour is the answer to one of the questions.

### Field log

Reproduce this in your `Lab2/README.md`. Fill it in while you are standing there.

| Field | open_standalone | open_rtk | occluded_standalone | occluded_rtk | walking_rtk |
|---|---|---|---|---|---|
| Date and start time | | | | | |
| Duration | | | | | |
| Location | | | | | |
| Mountpoint in use | n/a | | n/a | | |
| Time to first float (s) | n/a | | n/a | | |
| Time to first fix (s) | n/a | | n/a | | |
| Satellites, typical | | | | | |
| HDOP, typical | | | | | |
| Antenna height (m) | | | | | n/a |
| Weather | | | | | |
| Anything unusual | | | | | |

Time to first fix is graded and **cannot be reconstructed from the bag afterwards**.

### If corrections are not working

1. Is your phone hotspot on, and does the receiver show an IP address?
2. Are both NTRIP clients green in the Corrections tab? If red, re-check the caster
   address, the port, and the mountpoint
3. Try the backup mountpoint. Volunteer stations go offline, and `LFPWD_HV` being up
   when `BI_WARD_SANGALA` is down is exactly why you configured both
4. Check the RTK2go status page from a **laptop browser**, never by pointing a browser
   at the caster address itself
5. If you still cannot get corrections, record the standalone runs anyway and use the
   provided data set for the RTK half: [Lab 2 RTK on SharePoint](https://northeastern-my.sharepoint.com/:f:/r/personal/x_li1_northeastern_edu/Documents/EECE%205554%20Fall%202026/Lab%202%20RTK?d=w276d69cbc74e4b6bab79f17256d19562&csf=1&web=1&e=Cavrpb)

Because outing 2 is on your own schedule, a failed attempt costs you an afternoon
rather than the lab. Go back another day.

## Step 8: Analyze your data

Read the bags directly, as in Lab 1. For every run, work in UTM easting and northing,
in metres.

### Metrics

For each stationary run, compute the centroid of easting and northing and subtract it
from every point. Then report:

| Metric | Definition |
|---|---|
| Median centroid deviation | Median of the Euclidean distance from each point to the centroid |
| 95th percentile deviation | The upper tail. The distribution is skewed, so the mean hides it |
| Standard deviation, easting and northing | Reported separately |
| 2DRMS | `2 * sqrt(sigma_E^2 + sigma_N^2)`, the radius holding roughly 95 percent of fixes |
| Vertical standard deviation | On altitude |
| Fix quality distribution | Percentage of epochs at each value of `fix_quality` |

For `walking_rtk`, fit a line of best fit and report the root mean squared perpendicular
distance from the points to that line, as in Lab 1.

**Compute the metrics for `open_rtk` twice:** once over all epochs, and once over only
the epochs where `fix_quality == 4`. Report both. The difference is the cost of the float
epochs, and it is the number an engineer would actually ask you for.

> **numpy shape trap.** `(n,1) - (n,)` broadcasts to `(n,n)` silently and you get a
> giant array instead of a difference. Print `.shape` after every subtraction until you
> trust your arrays.

### Required plots

Every plot needs axis labels with units, and a legend when more than one data set is
shown. These are graded.

1. **Open sky scatter.** `open_standalone` and `open_rtk` on one figure, centroid
   subtracted, different markers. `ax.set_aspect('equal')` is **required**. Add a zoomed
   inset, because at a shared scale the RTK cloud collapses to a dot
2. **Occluded scatter.** `occluded_standalone` and `occluded_rtk`, same treatment, same
   axis limits as plot 1 so the two can be compared directly
3. **Histograms of distance from centroid.** Four subplots, one per stationary run
4. **Altitude versus time.** All five runs, plotted as offsets from each run's own mean
5. **Fix quality versus time.** `open_rtk`, `occluded_rtk` and `walking_rtk`, as a step
   plot, with time to first fix annotated on `open_rtk`
6. **Walking scatter with line of best fit.** Colour the points by `fix_quality` so the
   solution state is visible along the path

### Required table

| Metric | open_standalone | open_rtk (all) | open_rtk (fixed only) | occluded_standalone | occluded_rtk |
|---|---|---|---|---|---|
| Epochs | | | | | |
| Median centroid deviation (m) | | | | | |
| 95th percentile deviation (m) | | | | | |
| Std dev easting (m) | | | | | |
| Std dev northing (m) | | | | | |
| 2DRMS (m) | | | | | |
| Std dev altitude (m) | | | | | |
| Percent fix_quality 1 / 4 / 5 | | | | | |
| Mean satellites | | | | | |
| Mean HDOP | | | | | |

### Analysis questions

Answer these in `Lab2/README.md`. Two to four paragraphs each, referring to your own
numbers. A one-line answer earns nothing, and neither does a page of restated theory.

1. Your `open_standalone` centroid and your `open_rtk` centroid sit at different
   coordinates. Which is closer to the true position of your antenna? Explain what
   evidence would be needed to answer that, and why nothing you collected provides it.
2. Our primary reference station is about 13 km away. Which error sources does
   differencing remove and which does it not? Explain how the removed part degrades as
   the baseline grows, and estimate what you would expect at 200 km.
3. Using `open_rtk`, compare precision during float epochs against precision during
   fixed epochs. What quantity is resolved when a receiver moves from float to fixed,
   and why does that produce a discontinuous jump rather than a gradual improvement?
4. Compare the change from `open_standalone` to `occluded_standalone` against the change
   from `open_rtk` to `occluded_rtk`. Did RTK degrade proportionally more? Use your mean
   HDOP, your satellite counts, and your site photographs in the argument.
5. Compute the ratio of vertical to horizontal precision for each stationary run. The
   vertical is worse in all four. Explain the geometric reason, and say whether RTK
   changes the ratio.
6. Compare `walking_rtk` against `open_rtk`. How does the fix quality distribution
   differ, and how does the deviation from the line of best fit compare with the
   stationary centroid deviation? How much of the walking deviation is the receiver and
   how much is you not walking in a straight line?
7. A ground robot needs 5 cm horizontal accuracy at 10 Hz on a route that is half open
   parking lot and half tree-lined path. Based only on your measurements, is an RTK
   receiver of this type sufficient on its own? Justify with specific numbers from your
   table. If it is not, name what you would add and what that addition supplies that RTK
   does not.

Upload your six plots to the Lab 2 assignment on Canvas. Your written answers stay in
the repository.

## What to submit

The due date is posted on Canvas.

1. Push your `Lab2` folder to your private `EECE5554` repository
2. Upload your six plots to the Lab 2 assignment on Canvas
3. Confirm **ShawnLiNU** and the TAs are still collaborators on your repository

### Submission structure

<pre>
EECE5554
├── .gitignore
├── README.md                          (updated to describe Lab 2)
└── Lab2
    ├── README.md                      (field log, metrics table, and your seven answers)
    ├── [NortheasternUsername]_lab2.txt
    ├── analysis
    │   └── your analysis scripts
    ├── data
    │   ├── open_standalone
    │   ├── open_rtk
    │   ├── occluded_standalone
    │   ├── occluded_rtk
    │   └── walking_rtk
    ├── photos
    │   └── both sites, four directions each
    └── ros2_ws
        └── src
            ├── gps_msgs
            │   ├── CMakeLists.txt
            │   ├── msg
            │   │   └── Customrtk.msg
            │   └── package.xml
            └── gps_driver
                ├── gps_driver
                │   ├── __init__.py
                │   └── rtk_driver.py
                ├── launch
                │   └── rtk_driver.launch.py
                ├── package.xml
                ├── setup.cfg
                └── setup.py
</pre>

**Do not commit `build`, `install`, or `log`.** The self-check reports it if you do.

At 1 Hz these five bags are small, well under a megabyte each, so committing them is
fine. Run `git status` before committing and check what is actually staged.

## Assessment (120 points)

Two halves. Data collection is 40 points and data analysis is 80. The split reflects
where the effort goes: getting the receiver working is a real skill, but the analysis
is where you learn something.

### Data collection (40 pts)

**Driver and packages (14 pts)**
<ul>
    <li> <code>gps_msgs</code> builds with <code>Customrtk.msg</code> containing all thirteen fields (3 pts) </li>
    <li> <code>fix_quality</code> parsed from GGA field 6, not inferred from the scatter (3 pts) </li>
    <li> <code>num_satellites</code> and <code>correction_age</code> parsed, with empty fields handled (3 pts) </li>
    <li> <code>$GNGGA</code> accepted, and Lab 1 fields still correct (2 pts) </li>
    <li> <code>rtk_driver.launch.py</code> present and accepts the port argument (2 pts) </li>
    <li> Workspace builds cleanly, no build/install/log committed (1 pt) </li>
</ul>

**Receiver configuration (4 pts)**
<ul>
    <li> Verified during the classroom test: two correction sources connected, configuration saved and surviving a power cycle, <code>fix_quality</code> changing on <code>/gps</code> (4 pts) </li>
</ul>

**Data sets (16 pts)**
<ul>
    <li> Five rosbags present, correctly named, and of the stated durations (3 pts each for the four stationary runs, 4 pts for <code>walking_rtk</code>) </li>
</ul>

**Field record (6 pts)**
<ul>
    <li> Field log complete, including time to first float and time to first fix (4 pts) </li>
    <li> Site photographs, both sites, four directions each (2 pts) </li>
</ul>

### Data analysis (80 pts)

**Plots (40 pts)**
<ul>
    <li> Open sky scatter, centroid subtracted, equal aspect, with inset (7 pts) </li>
    <li> Occluded scatter, matching axis limits (7 pts) </li>
    <li> Histograms of distance from centroid, four subplots (7 pts) </li>
    <li> Altitude versus time, all five runs (6 pts) </li>
    <li> Fix quality versus time, with time to first fix annotated (6 pts) </li>
    <li> Walking scatter with line of best fit, coloured by fix quality (7 pts) </li>
</ul>

Each plot loses points for missing axis labels, missing units, or a missing legend when
more than one data set is shown.

**Metrics table (5 pts)**
<ul>
    <li> Complete, including the separate fixed-only column for <code>open_rtk</code> (5 pts) </li>
</ul>

**Analysis questions (35 pts)**
<ul>
    <li> Seven questions from Step 8, answered with your own numbers (5 pts each) </li>
</ul>

### Deductions that apply regardless of the rest

<ul>
    <li> Reporting centroid deviation as <b>accuracy</b>, or writing that RTK is "more accurate" without the caveat: <b>up to 8 points</b>. This lab measures precision, and the distinction is one of the learning objectives </li>
    <li> Scatter plots without equal aspect ratio: <b>4 points</b>. The visual comparison is meaningless without it </li>
</ul>

### If your field session does not work out

You can still complete the lab with the provided data set:
[Lab 2 RTK on SharePoint](https://northeastern-my.sharepoint.com/:f:/r/personal/x_li1_northeastern_edu/Documents/EECE%205554%20Fall%202026/Lab%202%20RTK?d=w276d69cbc74e4b6bab79f17256d19562&csf=1&web=1&e=Cavrpb). What you forfeit:

| Situation | Points lost | Maximum |
|---|---|---|
| Collected everything yourself | 0 | 120 |
| Collected the standalone runs, used provided RTK data | 14 | 106 |
| Used the provided data set entirely | 22 | 98 |

Nothing on the analysis side is affected. State clearly in your `README.md` which runs
are yours and which are provided. Presenting provided data as your own is an academic
integrity matter.

### If something is not working

Push the self-check report exactly as generated, failures included, and describe in
your `README.md` the exact error and your plan. An honest, specific failure report on
time earns substantial partial credit. A hand-edited report claiming results the log
does not show earns zero. The grader cross-checks the report against what is committed.

Note that the driver is only 14 of 120 points. A driver with one broken field still
lets you collect data and earn most of the analysis marks, so do not stall on it.

## Appendix A: Common problems

### Configuration

**Browser cannot reach `192.168.3.1`**
The receiver is still booting, or you have a charge-only USB-C cable. Wait 60 seconds,
run `ip a` to confirm a new network interface appeared, then try a different cable.
Note that macOS does not reliably bind a driver to this interface. Use your Ubuntu
machine, which is the documented configuration and what you will use in the field.

**Mount point dropdown stays empty**
The receiver has no network path to the caster. Go back to the hotspot step. The
dropdown is populated by fetching the caster's stream list, so it cannot fill in until
the receiver itself is online.

**NTRIP client line is red**
Wrong caster address, wrong port, or no network. Check those three in that order.

**Settings vanish after a power cycle**
The configuration was never saved. Redo it and test the power cycle again before you
leave the classroom.

### Data flow

**No `/dev/ttyACM*` device**
Not powered, or a charge-only cable. Try another USB-C cable.

**Permission denied on the serial port**
You are not in the `dialout` group, or you have not logged out since being added. Check
with `id -nG`. Do not use `chmod 777`.

**`/gps` publishes nothing, but the web interface looks fine**
NMEA output is not enabled on the USB port, or your launch file is pointing at the wrong
device. Check the web interface output settings first.

**`ValueError: could not convert string to float`**
An empty NMEA field. Use the `to_float` helper from Step 3 on every numeric field, not
just `correction_age`.

**The driver worked in Lab 1 and reads nothing now**
You are matching `$GPGGA` only. The mosaic-X5 emits `$GNGGA`.

### Solution quality

**Web interface says RTK fixed, but `/gps` reads 1**
Your driver is reading a different serial port from the one carrying NMEA. The receiver
exposes several.

**`fix_quality` stuck at 1 with both clients green**
The receiver is getting RTCM it cannot use. Note it in your field log and carry on. This
is a reportable result, not necessarily your mistake.

**Reaches 5 but never 4**
Poor sky view, or too few satellites for ambiguity resolution. Expected at the occluded
site. At the open site, move away from trees and wait longer.

**Fix drops every few seconds**
Corrections are arriving intermittently. Watch `correction_age`. Above roughly 30
seconds the receiver reverts to standalone.

**The RTK cloud is invisible on the scatter plot**
Correct and expected. Add a zoomed inset. Do not change the aspect ratio to compensate,
which costs 4 points.

**`ros2 bag record` produces an empty bag**
You started recording before the driver was publishing. Confirm with
`ros2 topic hz /gps` first.